In [ ]:
!pip install ydata-profiling
import pandas as pd
from ydata_profiling import ProfileReport
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import calendar

## 1. Load Data

In [ ]:
customer_df = pd.read_csv("/content/UK-Bank-Customers.csv")

## 2. Data Overview

In [ ]:
print(f"Number of Rows: {customer_df.shape[0]}")
print(f"Number of Columns : {customer_df.shape[1]}")
print(f"Head :\n{customer_df.head()}")

In [ ]:
print(customer_df.info())
print(f"Describe:\n{customer_df.describe()}")

In [ ]:
print(f"Columns Name :\n {customer_df.columns.to_list()}")

In [ ]:
profile = ProfileReport(customer_df, title='UK Bank Customers Profiling Report')

In [ ]:
profile.to_notebook_iframe()

##3. Data Cleaning

In [ ]:
missing_values = customer_df.isnull().sum()
missing_percentage = (missing_values / len(customer_df)) * 100
missing_report = pd.DataFrame({"Missing Count" : missing_values, "Missing Percentage" : missing_percentage})
print(f'Missing Report:\n{missing_report.sort_values(by="Missing Percentage")}')

In [ ]:
customer_df.duplicated().sum()

In [ ]:
customer_df["Age"].between(18,100).all()

In [ ]:
# Convert Date Joined Column to Datetime
customer_df["Date Joined"] = pd.to_datetime(customer_df["Date Joined"], format = "%d.%b.%y" )
print(customer_df["Date Joined"])

# Select Numerical,  Categorical, Date Time

#Numerical
customer_df_num = customer_df[["Age", "Balance"]]
print(customer_df_num.head())

#Time
customer_df_time = customer_df[["Date Joined"]]
print(customer_df_time.head())

#Categorical
customer_df_cat = customer_df.drop(["Age", "Balance", "Date Joined"], axis=1)
print(customer_df_cat.head())

customer_df.hist(bins=50,figsize=(20, 15))
plt.show()


In [ ]:
#Describe Statistics of Numerical Values

for index in range(len(customer_df_num.columns)):
     col = customer_df_num.columns[index]
     print(f'Describe of {col} :\n{customer_df_num[col].describe()}')

customer_df["Join_Month"] = customer_df["Date Joined"].dt.month
print(f"Join Month:\n {customer_df['Join_Month'].head()}")

customer_df["Join_Quarter"] = customer_df["Date Joined"].dt.quarter
print(f"Join Quarter:\n {customer_df['Join_Quarter'].head()}")

print(f"First Join Date:\n {customer_df['Date Joined'].min()}")
print(f"Last Join Date:\n {customer_df['Date Joined'].max()}")


In [ ]:
# Outlier Detection
Q1_age = customer_df["Age"].quantile(0.25)
Q3_age = customer_df["Age"].quantile(0.75)
IQR_age = Q3_age - Q1_age

lower_age = IQR_age - 1.5*IQR_age
upper_age = IQR_age + 1.5*IQR_age

age_outliers = customer_df [
    (customer_df["Age"] < lower_age) |
    (customer_df["Age"] > upper_age)
]

age_outliers_percentage = (
    len(age_outliers) / len(customer_df)
)*100

print(f"Number of Age Outliers: \n{len(age_outliers)}")
print(f"Age Outliers: \n {age_outliers_percentage}")

Q1_balance = customer_df["Balance"].quantile(0.25)
Q3_balance = customer_df["Balance"].quantile(0.75)

IQR_balance = Q3_balance - Q1_balance

lower_balance = IQR_balance - 1.5*IQR_balance
upper_balance = IQR_balance + 1.5*IQR_balance

balance_outliers = customer_df [
    (customer_df["Balance"] < lower_balance) |
    (customer_df["Balance"] > upper_balance)
]

balance_outliers_percentage = (
    len(balance_outliers) / len(customer_df)
)*100


print(f"Number of Balance Outliers: \n{len(balance_outliers)}")
print(f"Balance Outliers: \n {balance_outliers_percentage}")

##4. Feature Engineering

In [ ]:

#Classification Age
def age_group(age):
     if age<= 24:
          return "18-24"
     elif 24<age<=44:
          return "25-44"
     elif 44<age<=59:
          return "45-59"
     else :
          return "+60"

customer_df["Age_Group"] = customer_df["Age"].apply(age_group)

counts_gender_cls_age = customer_df.groupby(["Age_Group", "Gender"]).size().reset_index(name="Count")

order_age_clf = ["18-24", "25-44", "45-59", "+60"]
counts_gender_cls_age["Age_Group"]= pd.Categorical(counts_gender_cls_age["Age_Group"], categories=order_age_clf, ordered=True)

new_featueres = ["Age_Group", "Join_Month", "Join_Quarter"]
print(customer_df[new_featueres].head(10))


In [ ]:
median_blnc_by_age = customer_df.groupby("Age_Group")["Balance"].median().reset_index()
print(f'Median Balance by Age:\n{median_blnc_by_age}')

In [ ]:
job_region_blnc = customer_df.groupby(["Job Classification", "Region"])["Balance"].median().unstack()
print(job_region_blnc)

In [ ]:
join_by_month =pd.DataFrame(customer_df['Date Joined'].dt.month).value_counts().sort_index().reset_index()
join_by_month.columns = ['Date Joined', 'Customer Count']
join_by_month['Month Name'] = join_by_month['Date Joined'].apply(lambda x: calendar.month_name[x])

In [ ]:
region_count = customer_df['Region'].value_counts().reset_index()
region_count.columns = ['Region Name', 'Region Count']

In [ ]:
blnc_by_job = customer_df.groupby('Job Classification')['Balance'].agg(['mean', 'std', 'median', 'count', 'min', 'max']).reset_index()
print(blnc_by_job)


In [ ]:
region_blnc = customer_df.groupby('Region')['Balance'].agg(
    Mean_Balance = 'mean',
    Std_Balance = 'std',
    Median_balance = 'median',
    Count ='count',
    Min = 'min',
    Max = 'max' ).reset_index()
print(region_blnc)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
custom_palette = ['#003049', '#669BBC', '#D6CCC2', '#9E9E9E', '#1E88E5']
main_text = '#003049'
primary_bar = '#669BBC'
secondary_highlight = '#D6CCC2'
background = '#9E9E9E'
accent_line = '#1E88E5'

sns.set_theme(
    context="talk",
    style="darkgrid",
    palette=custom_palette
)

figsize = (10,5)

plt.rcParams.update({
     "figure.figsize": (10,5),
     "axes.titlesize": 16,
     "axes.titleweight": "bold",
     "axes.labelsize": 14,
     "axes.labelweight": "bold",
     "xtick.labelsize": 12,
     "ytick.labelsize": 12,
     "legend.fontsize": 12,
     "legend.title_fontsize": 13,
     "font.family": "DejaVu Sans",
     "axes.spines.top": False,
     "axes.spines.right": False
})

## 5.1 Median Balance by Age Clasification

In [ ]:
fig, ax = plt.subplots(figsize= figsize)

sns.barplot(
     data=counts_gender_cls_age,
     x="Age_Group",
     y="Count",
     hue="Gender",
     palette=[custom_palette[1], custom_palette[2]],
     ax=ax,
     edgecolor="black",
     linewidth=1.0

     )

for container in ax.containers:
     ax.bar_label(container, fmt="%d", label_type="edge", padding=3, fontsize=11, fontweight="bold")

ax.set_xlabel("Age Classification")
ax.set_ylabel("Number of Customer")
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.grid(axis='x', visible=False)
leg = ax.legend(
    title="Gender",
    frameon=True,
    fancybox=True,
    shadow=False,
    loc="upper right"
)


ax2 = ax.twinx()
sns.lineplot(
     data= median_blnc_by_age,
     x= "Age_Group",
     y= "Balance",
     ax= ax2,
     color= custom_palette[0],
     markers= "o",
     linewidth= 1,
     label= "Median Balance"
)

ax2.set_ylabel("Median Balance")
ax2.grid(False)
leg = ax2.legend(
    frameon=True,
    fancybox=True,
    shadow=False,
    loc="upper left"
)

plt.title("Distribution of Age Groups and Median Balance")
leg.get_frame().set_edgecolor("gray")
leg.get_frame().set_linewidth(0.8)
plt.tight_layout()
plt.savefig(
    "Distribution of Age Groups and  Median Balance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Insight :
The majority of customers belong to the 25–44 age group, making it the largest customer segment for the bank across both genders.
The 45–59 age group has fewer customers compared to the 25–44 group, but it shows a higher median account balance, indicating greater financial value per customer.
Customers aged 60+ have the highest median balance, despite representing the smallest customer segment, suggesting that older customers tend to maintain higher account balances.
The customer distribution between males and females is relatively balanced in most age groups, with a slightly higher number of male customers in the older age groups.

## 5.2 Balance Distribution by Job Classifiacation

In [ ]:
plt.figure(figsize= figsize)

ax = sns.boxplot(
    data = customer_df,
    x = "Job Classification" ,
    y = "Balance",
    palette = custom_palette
    )

plt.title("Balance Distribution by Job Classification")
plt.xlabel("Job Classification")
plt.ylabel("Balance")
plt.xticks(rotation=45)
plt.savefig(
    "Balance Distribution by Job Classification.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Insight :
The distribution of account balances is relatively similar across all job classifications, with no major difference in median balance between White Collar, Blue Collar, and Other customers.
Customers in the Other job classification show a slightly higher median balance compared to the other groups, indicating a potentially higher average financial position.
All job classifications contain a considerable number of high-balance outliers, suggesting that a small group of customers maintain significantly larger account balances regardless of their job category.
The large spread and presence of outliers indicate that job classification alone may not be a strong factor for explaining customer account balance variations.

## 5.3 Balance Distribution by Region

In [ ]:
plt.figure(figsize= figsize)

ax = sns.boxplot(
    data = customer_df,
    x = "Region" ,
    y = "Balance",
    palette = custom_palette
    )

plt.title("Balance Distribution by Region")
plt.xlabel("Region")
plt.ylabel("Balance")
plt.xticks(rotation=45)
plt.savefig(
    "Balance Distribution by Region.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Insight :
The median account balance is relatively similar across all regions, indicating no substantial regional differences in customer balances.
Customers from Wales appear to have a slightly higher median balance compared to the other regions.
All regions exhibit a wide spread of account balances and contain several high-value outliers, suggesting that high-balance customers are present throughout the UK.
The similarity in balance distributions implies that region alone is unlikely to be a strong predictor of customer account balance.

## 5.4 Balance Distribution by Age Group

In [ ]:
plt.figure(figsize=figsize)
sns.boxplot(
     data= customer_df,
     x = "Age_Group",
     y = "Balance",
     order = order_age_clf,
     palette = custom_palette,
     hue = "Age_Group",
     legend = False
)

plt.title("Balance Distribution by Age Group")
plt.savefig(
    "Balance Distribution by Age.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Insight: The median account balance generally increases from the younger age groups to the 45–59 age group, indicating that older customers tend to maintain higher account balances.
The 45–59 and 60+ age groups exhibit a wider spread in account balances, suggesting greater variability in customers' financial status.
High-balance outliers are present across all age groups, indicating that customers with exceptionally large account balances exist regardless of age.
Although younger customers (18–24) have lower median balances, some individuals still maintain significantly high account balances.

## 5.5 Median Balance by Job Classification and Region




In [ ]:
plt.figure(figsize= figsize)
sns.heatmap(
     data = job_region_blnc,
    annot=True,
    fmt=".0f",
    annot_kws={"size":8},
    cbar_kws={"label": "Median  Balance"},
    cmap=custom_palette,
    linewidths=0.5,
    linecolor="white"
)

plt.title("Median Balance by Job Classification and Region")
plt.savefig(
    "Median Balance by Job Classification and Region.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


Insight : Median account balances vary moderately across different combinations of job classification and region.
Wales has the highest median balance for both Blue Collar and White Collar customers.
Among customers classified as Other, England records the highest median balance.
No single region consistently has the highest median balance across all job classifications, suggesting that customer balance patterns are influenced by the combination of both region and job classification rather than either factor alone.

## 5.6 Join Month Analysis

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(
     data = join_by_month,
     x = "Month Name",
     y = "Customer Count",
     color = custom_palette[0],
     edgecolor="black",
     linewidth=1.0
)

for container in ax.containers:
    ax.bar_label(container, fontsize=10)

plt.title("Customer Acquisition by Month")
plt.xlabel("Month")
plt.ylabel("Number of Customers")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    "Customer Acquisition by Month.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Insight: Customer acquisition shows an overall increasing trend throughout the year.
The first quarter (January to March) records the lowest number of new customers.
Customer acquisition begins to increase from April and reaches its highest levels during the final quarter of the year.
October and December have the highest customer acquisition, suggesting that customer onboarding activity is stronger toward the end of the year.
The observed pattern may indicate seasonal effects or the impact of marketing campaigns; however, additional business information would be required to determine the exact cause.

## 5.7 Region analysis

In [ ]:
plt.figure(figsize=(10,6))

ax = sns.barplot(
    data = region_count,
    x = 'Region Name',
    y = 'Region Count',
    palette = custom_palette,
    edgecolor="black",
    linewidth=1.0
)
for i, row  in region_count.iterrows():
    ax.text(
        i,
        row["Region Count"] + 30,
        f'{row["Region Count"]}\n({row["Percentage"]}%)',
        ha="center",
        fontsize=10,
        fontweight="bold"
    )
plt.title("Customer Distribution by Region")
plt.xlabel("Region")
plt.ylabel("Number of Customers")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    "Customer Distribution by Region.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

Insight: England represents the largest customer base, accounting for 53.8% of all customers (2,159 customers), indicating that the bank's operations are heavily concentrated in this region.
Scotland is the second-largest market with 28.0% of customers, while Wales (13.0%) and Northern Ireland (5.3%) contribute considerably smaller shares.
The customer distribution is uneven across regions, suggesting that the bank has a stronger market presence in England than in the other UK regions.
The relatively small customer bases in Wales and Northern Ireland may indicate potential opportunities for future customer acquisition and regional expansion.